<a href="https://colab.research.google.com/github/quiquefluque/Python-Finace/blob/main/Task_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yfinance as yf

#Carga y limpieza de datos
path="/content/drive/MyDrive/datasheet/Nat_Gas (1).csv"
datos=pd.read_csv(path)
df=pd.DataFrame(datos)

#Busca el precio del gas en la tabla, para la fecha que quiera
def get_price(date):#date es un dato que aporta el usuario
    price = df.loc[df['Dates'] == date, 'Prices'].values #de las filas en que dates=date, dame solo los precios, . values devuelve una lista
    return price[0] if len(price) > 0 else None #devuelve el primero de la lista si no esta vacia

#Contrato, parametros que vienen de fuera función, i_d-cuando compro,w_d-cuando vendo,rate-cunato gas muevo cada vez,i_c-cuanto cuesta mover,max_vol limite alm,storage_fee- lo que cuesta almacenar
def price_contract(injection_dates, withdrawal_dates, rate, max_vol, storage_fee, injection_cost):
    balance = 0 #cuenta del banco
    inventory = 0 #tanque de gas

    # 1. Procesar Compras (Inyecciones)
    for date in injection_dates:#para cada fecha en la lista de fechas de compra
        price = get_price(date)
        if inventory + rate <= max_vol:
            # Costo = (Gas * Precio) + Costo de meterlo
            cost = (rate * price) + (rate * injection_cost)
            balance -= cost
            inventory += rate
        else:
            print(f"Error: No hay espacio para inyectar en {date}")

    # 2. Procesar Ventas (Retiros)
    for date in withdrawal_dates:
        price = get_price(date)
        if inventory >= rate:
            # Ganancia = (Gas * Precio) - Costo de sacarlo
            revenue = (rate * price) - (rate * injection_cost)
            balance += revenue
            inventory -= rate
        else:
            print(f"Error: No hay gas suficiente para retirar en {date}")

    # 3. Restar Alquiler (Storage)
    # Ejemplo: Si dura 6 meses, restamos 6 * storage_fee
    total_months = (withdrawal_dates[-1] - injection_dates[0]).days // 30
    balance -= total_months * storage_fee

    return balance

# ==========================================
# SOLICITUD DE INPUTS AL USUARIO
# ==========================================
print(">>> SIMULADOR DE CONTRATOS JP MORGAN <<<")

try:
    # Pedir datos numéricos
    max_vol = float(input("Capacidad máxima del almacén (ej. 5000000): "))
    rate = float(input("Cantidad de gas por operación (rate): "))
    storage_fee = float(input("Costo de alquiler mensual fijo: "))
    injection_cost = float(input("Costo operativo por unidad (inyección/retiro): "))

    # Pedir fechas y convertirlas a objetos datetime
    # El usuario debe usar formato YYYY-MM-DD para evitar errores
    print("\n* Ingresa las fechas en formato AAAA-MM-DD (ej. 2021-06-30) *")

    inj_input = input("Fechas de INYECCIÓN (separadas por comas): ")
    # Convertimos cada fecha de texto a objeto datetime
    inj_dates = [pd.to_datetime(d.strip()) for d in inj_input.split(',')]

    with_input = input("Fechas de RETIRO (separadas por comas): ")
    with_dates = [pd.to_datetime(d.strip()) for d in with_input.split(',')]

    # Ejecutar el modelo
    resultado = price_contract(inj_dates, with_dates, rate, max_vol, storage_fee, injection_cost)

    print(f"\n==========================================")
    print(f"VALOR NETO DEL CONTRATO: ${resultado:,.2f}")
    print(f"==========================================")

except Exception as e:
    print(f"Error en los datos ingresados: {e}")

>>> SIMULADOR DE CONTRATOS JP MORGAN <<<
Capacidad máxima del almacén (ej. 5000000): 100000
Cantidad de gas por operación (rate): 10
Costo de alquiler mensual fijo: 100000
Costo operativo por unidad (inyección/retiro): 20

* Ingresa las fechas en formato AAAA-MM-DD (ej. 2021-06-30) *
Fechas de INYECCIÓN (separadas por comas): 2020-04-12
Fechas de RETIRO (separadas por comas): 2222-05-12
Error en los datos ingresados: unsupported operand type(s) for *: 'float' and 'NoneType'
